# Bronze → Silver Transformation

Reads raw Parquet from the **Bronze** layer, converts all `date`/`Date` columns from
datetime to `yyyy-MM-dd` string, and writes the result as **Delta** to the Silver layer.

> **Requires:** Run `storagemount` first (or ensure OAuth2 Spark config is set on the cluster).

In [ ]:
# ── Auth & path config (runs storagemount notebook to set OAuth2 + path vars) ──
%run ./storagemount

## 1. Verify Bronze contents

In [ ]:
display(dbutils.fs.ls(f"{BRONZE}/SalesLT/"))

## 2. Single-table preview (Address)

In [ ]:
df = spark.read.format('parquet').load(f"{BRONZE}/SalesLT/Address/Address.parquet")
display(df)

In [ ]:
from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType

df = df.withColumn(
    "ModifiedDate",
    date_format(from_utc_timestamp(df['ModifiedDate'].cast(TimestampType()), "UTC"), "yyyy-MM-dd")
)
display(df)

## 3. Transform all SalesLT tables → Silver

In [ ]:
from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType

# Discover all tables from bronze
table_names = [f.name.split('/')[0] for f in dbutils.fs.ls(f"{BRONZE}/SalesLT/")]
print("Tables found:", table_names)

for table in table_names:
    input_path  = f"{BRONZE}/SalesLT/{table}/{table}.parquet"
    output_path = f"{SILVER}/SalesLT/{table}/"

    df = spark.read.format('parquet').load(input_path)

    # Convert any date/Date column from datetime → yyyy-MM-dd
    for col_name in df.columns:
        if "Date" in col_name or "date" in col_name:
            df = df.withColumn(
                col_name,
                date_format(from_utc_timestamp(df[col_name].cast(TimestampType()), "UTC"), "yyyy-MM-dd")
            )

    df.write.format('delta').mode('overwrite').save(output_path)
    print(f"  ✅ {table} → {output_path}")

print("\n✅ Bronze → Silver complete")

## 4. Verify Silver output

In [ ]:
display(dbutils.fs.ls(f"{SILVER}/SalesLT/"))

In [ ]:
# Spot-check: preview last processed table
display(df)